# Wilson-Cowan network walkthrough

A step-by-step tour of `src/model.py`. Goal: understand what the model is, see it produce an alpha-band rhythm, and build the lookup table that maps the inhibitory time constant `tau_inh` to the network's peak frequency.

**What is a Wilson-Cowan unit?** A pair of two interacting neural populations: excitatory (E) and inhibitory (I). E excites itself and I, I inhibits E and itself. With the right parameters the pair oscillates, much like a predator-prey system. One unit represents one chunk of cortex.

**Why a network of units?** The assignment asks for a dynamic *network* model. We connect N such units with an all-to-all coupling and a single global strength. The free parameter we vary is `tau_inh`.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import welch

# Make src/ importable
REPO_ROOT = Path("..").resolve()
sys.path.insert(0, str(REPO_ROOT))

from src.model import (
    build_network,
    run_and_get_excitatory,
    peak_frequency,
    simulate_peak_frequency,
)

## 2. A single Wilson-Cowan unit

Start with N=1 to see what the model output looks like. We pick `tau_inh = 20 ms` which (from our prior sweep) gives an alpha-range oscillation.

In [ ]:
model = build_network(n_nodes=1, tau_inh=20.0)
t, exc = run_and_get_excitatory(model)
print("exc shape:", exc.shape)             # (1, n_samples)
print("Duration:", t[-1], "ms")
print("dt:", model.params['dt'], "ms")

### Time series

Plot the excitatory activity over a 1-second window after the transient. You should see a clean oscillation.

In [ ]:
mask = (t > 3000) & (t < 4000)   # 1 s window, well past transient
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(t[mask] / 1000.0, exc[0, mask], color="#0f766e")
ax.set_xlabel("time (s)")
ax.set_ylabel("excitatory activity")
ax.set_title("Wilson-Cowan single unit, tau_inh = 20 ms")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

### Power spectrum

Compute the PSD of the excitatory signal (skipping the first 2 s as transient) and find the peak frequency in the 3-25 Hz band.

In [ ]:
dt = model.params['dt']
fs = 1000.0 / dt
sig = exc[0, int(2000/dt):]      # skip 2 s
sig = sig - sig.mean()
freqs, psd = welch(sig, fs=fs, nperseg=int(2 * fs))

peak_hz = peak_frequency(exc[0], dt_ms=dt)
print(f"Peak frequency: {peak_hz:.2f} Hz")

fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogy(freqs, psd, color="#0f766e")
ax.axvspan(8, 13, color="grey", alpha=0.15, label="alpha (8-13 Hz)")
ax.axvline(peak_hz, color="#92400e", linestyle="--",
           label=f"peak = {peak_hz:.1f} Hz")
ax.set_xlim(0, 30)
ax.set_xlabel("frequency (Hz)")
ax.set_ylabel("PSD")
ax.set_title("Single-unit PSD, tau_inh = 20 ms")
ax.legend()
plt.tight_layout()
plt.show()

## 3. Sweep `tau_inh` and build the lookup table

For a range of `tau_inh` values, run the model and record the peak frequency. The resulting curve is exactly the "model lookup table" that the per-subject fitting step will invert.

In [ ]:
tau_grid = np.arange(11, 30, 0.5)   # ms
peaks_1node = np.array([simulate_peak_frequency(1, t) for t in tau_grid])

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(tau_grid, peaks_1node, "o-", color="#0f766e", label="N = 1")
ax.axhspan(9, 11, color="#0f766e", alpha=0.08, label="Control alpha range")
ax.axhspan(7, 9, color="#92400e", alpha=0.08, label="PD alpha range")
ax.set_xlabel("tau_inh (ms)")
ax.set_ylabel("peak frequency (Hz)")
ax.set_title("Peak frequency vs tau_inh (single node)")
ax.grid(alpha=0.25)
ax.legend()
plt.tight_layout()
plt.show()

## 4. Network of N = 8 nodes

Now repeat with N = 8 all-to-all coupled units, the configuration we plan to use in the actual pipeline. All nodes share the same `tau_inh`, so by symmetry they synchronize.

In [ ]:
model8 = build_network(n_nodes=8, tau_inh=20.0)
t8, exc8 = run_and_get_excitatory(model8)
print("exc8 shape:", exc8.shape)   # (8, n_samples)

# All eight node time series, overlaid; they should overlap completely
mask = (t8 > 3000) & (t8 < 4000)
fig, ax = plt.subplots(figsize=(8, 3))
for i in range(8):
    ax.plot(t8[mask] / 1000.0, exc8[i, mask], alpha=0.6, lw=1)
ax.set_xlabel("time (s)")
ax.set_ylabel("excitatory activity")
ax.set_title("8-node Wilson-Cowan network, tau_inh = 20 ms (nodes synchronize)")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 5. Compare N = 1 vs N = 8 curves

The lookup table the next script (`scripts/02_simulate_network.py`) will save is the N = 8 version. We expect it to be similar in shape to the N = 1 curve, with a small frequency offset from the coupling.

In [ ]:
peaks_8node = np.array([simulate_peak_frequency(8, t) for t in tau_grid])

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(tau_grid, peaks_1node, "o-", color="#0f766e", label="N = 1")
ax.plot(tau_grid, peaks_8node, "s-", color="#92400e", label="N = 8")
ax.axhspan(9, 11, color="grey", alpha=0.10, label="Control alpha range")
ax.axhspan(7, 9, color="grey", alpha=0.05, label="PD alpha range")
ax.set_xlabel("tau_inh (ms)")
ax.set_ylabel("peak frequency (Hz)")
ax.set_title("Peak frequency vs tau_inh: single unit vs 8-node network")
ax.grid(alpha=0.25)
ax.legend()
plt.tight_layout()
plt.show()

## 6. What's next

This notebook validates that `src/model.py` produces a clean alpha-range oscillation with `tau_inh` as a monotonic control knob. The next pipeline step (`scripts/02_simulate_network.py`) will:

1. Run the N = 8 sweep above on a finer `tau_inh` grid.
2. Save the (`tau_inh`, peak frequency) pairs to `results/peak_curve.csv`.

Then `scripts/03_fit_per_subject.py` will use this CSV as a lookup table: for each subject's measured alpha peak (from `features.csv`), it will find the `tau_inh` that the model would need to reproduce that peak. The distribution of fitted `tau_inh` values between PD and Control is the model-level test of the hypothesis.